# BASA — Audit Kebocoran Data (Near-Duplicate)**Menjawab catatan promotor #4: "Pikirkan kebocoran data."**Status sebelum notebook ini:- **Nitik / kebocoran rotasi** — SUDAH diuji. LEAKY − GROUP = 0 (semua 100%, std 0). Bukan kebocoran rotasi.- **Near-duplicate pada dataset rakitan** — BELUM pernah diukur. §4.4 paper hanya menyebutnya sebagai  caveat kualitatif: *"web-assembled datasets may contain near-duplicate images... this does not  affect our retention ratios"*. Itu klaim **tanpa data**. Notebook ini yang mengisinya.Keluaran notebook:| Berkas | Guna ||---|---|| `*_summary.csv` | Laju duplikat → menggantikan caveat kualitatif §4.4 || `*_leak_under_paper_split.csv` | % citra uji yang punya kembaran di train, per seed → **angka utama untuk #4** || `*_crossclass_pairs.csv` | Citra ~sama berlabel BEDA → kontaminasi jenis baru, sambungan ke §3.1 || `*_grouped_splits.csv` | Split bebas-bocor, siap dipakai ulang di kode training |**Waktu:** ±5–15 menit, CPU saja. Tidak perlu GPU (matikan akselerator agar kuota tidak terpakai).---### Catatan metode: mengapa BUKAN perceptual hashCara standar deteksi near-duplicate adalah pHash. **Itu salah untuk batik.** Batik adalah tekstur**periodik**; pada citra periodik ambang median DCT di pHash tidak stabil (koefisien menumpuk disekitar median, sehingga perubahan kecil membalik banyak bit). Pada uji offline bermotif periodik:- **False positive:** 6 dari 9 pasangan lolos pHash ≤ 5 padahal korelasi piksel ~0,0- **False negative:** pasangan dengan korelasi **1,000** (duplikat sempurna) berjarak pHash **9, 10, 18** → terlewatNotebook ini memakai **korelasi piksel eksak** via satu matmul (~1 detik untuk 5.000 citra),plus pencocokan **dihedral** (8 rotasi/cerminan) untuk menangkap duplikat yang terputar.Tervalidasi **100% recall, 100% precision** terhadap duplikat ground-truth (Sel 5).Simpan alasan ini — ini jawaban siap-pakai bila reviewer bertanya kenapa tidak pakai pHash.

## Sel 1 — Inspeksi struktur (WAJIB DIJALANKAN DULU)**Jangan lewati sel ini.** Sel ini tidak menghitung apa pun; ia hanya memastikan path benar dan**label diambil dari tempat yang benar**.Ini penting karena ketiga dataset berbeda strukturnya. Menurut catatan, **Nitik berupa folderDATAR dengan kelas di NAMA FILE** (`1 Sekar Kemuning 1_rotate_180.jpg`), bukan di folder induk.Kalau label diambil dari folder induk, seluruh 960 citra akan berlabel sama dan angkalintas-kelas menjadi tak bermakna.Periksa keluaran di bawah: **kolom `n_kelas` harus masuk akal** (Nitik 60, DION 20, Jambi 5).Kalau tidak, betulkan `ROOT`/`LABEL_MODE` sebelum lanjut.

In [ ]:
import os, re, sys, hashlib
from pathlib import Path
from collections import Counter

EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

DATASETS = {
    "nitik": dict(
        root="/kaggle/input/datasets/hertiyani/batik-nitik-960-official-mendeley/Batik Nitik 960",
        label_mode="nitik_filename",   # kelas dari NAMA FILE, bukan folder
        expect_n=960, expect_k=60,
    ),
    "dion": dict(
        root="/kaggle/input/datasets/dionisiusdh/indonesian-batik-motifs",
        label_mode="parent",           # kelas = nama folder induk
        expect_n=983, expect_k=20,
    ),
    "jambi": dict(
        root="/kaggle/input/datasets/hertiyani/batik-jambi/batik jambi",
        label_mode="parent",
        expect_n=161, expect_k=5,
    ),
}

def parse_nitik(stem):
    """<classnum> <MotifName> <instancenum>[_rotate_XXX]
    contoh: '1 Sekar Kemuning 1_rotate_180'; tangani spasi-ganda '31 Krawitan  2'."""
    rot = 0
    m = re.search(r"_rotate_(\d+)", stem)
    if m:
        rot = int(m.group(1)); stem = re.sub(r"_rotate_\d+", "", stem)
    toks = re.sub(r"\s+", " ", stem).strip().split(" ")
    return dict(classnum=toks[0], instnum=toks[-1],
                motif=" ".join(toks[1:-1]), rotation=rot)

def label_of(p, mode):
    if mode == "parent":
        return p.parent.name
    if mode == "nitik_filename":
        r = parse_nitik(p.stem)
        return f"{r['classnum']} {r['motif']}".strip()
    raise ValueError(mode)

print("="*78)
for name, cfg in DATASETS.items():
    root = Path(cfg["root"])
    print(f"\n### {name.upper()}  ->  {root}")
    if not root.exists():
        print("   [FATAL] path TIDAK ADA. Cek panel 'Input' di kanan, betulkan ROOT.")
        # bantu cari
        base = Path("/kaggle/input")
        if base.exists():
            print("   Kandidat yang tersedia:")
            for d in sorted(base.glob("*/*"))[:20]:
                print("     ", d)
        continue

    files = sorted(p for p in root.rglob("*") if p.suffix.lower() in EXTS)
    labels = [label_of(p, cfg["label_mode"]) for p in files]
    ks = Counter(labels)

    ok_n = "OK" if len(files) == cfg["expect_n"] else f"<< BEDA (paper: {cfg['expect_n']})"
    ok_k = "OK" if len(ks) == cfg["expect_k"] else f"<< BEDA (paper: {cfg['expect_k']})"
    print(f"   n_citra : {len(files):5d}   {ok_n}")
    print(f"   n_kelas : {len(ks):5d}   {ok_k}   (mode label = {cfg['label_mode']})")
    print(f"   contoh nama file : {[p.name for p in files[:3]]}")
    print(f"   contoh label     : {labels[:3]}")
    print(f"   kelas terkecil/terbesar : {min(ks.values())} / {max(ks.values())} citra")

    # kedalaman folder -> bantu deteksi salah tebak struktur
    depths = Counter(len(p.relative_to(root).parts) for p in files)
    print(f"   kedalaman relatif: {dict(depths)}  (1 = folder datar)")

    if cfg["label_mode"] == "nitik_filename":
        inst = {f"{parse_nitik(p.stem)['classnum']}|{parse_nitik(p.stem)['instnum']}" for p in files}
        rots = Counter(parse_nitik(p.stem)["rotation"] for p in files)
        print(f"   instance unik : {len(inst)}  (harap 240 = 60 kelas x 4 instance)")
        print(f"   sebaran rotasi: {dict(sorted(rots.items()))}  (harap 240 tiap sudut)")
print("\n" + "="*78)
print("BACA KELUARAN DI ATAS. Lanjut HANYA bila n_kelas sudah masuk akal.")

## Sel 2 — Modul auditMandiri, tanpa dependensi tambahan (numpy/pandas/pillow/scipy sudah ada di Kaggle).**Dua definisi yang sengaja dipisah — jangan dicampur:**| | Dipakai untuk | Sifat ||---|---|---|| **Pasangan langsung** (edge) | **Semua angka yang diklaim di paper** | Tiap pasangan benar-benar berkorelasi ≥ ambang || **Klaster transitif** (union-find) | **Hanya membuat split** | Over-grouping = konservatif, tak ada kembaran lolos |Klaster berantai (A~B, B~C, tapi A≁C) **tidak boleh** dilaporkan sebagai duplikat: reviewer yangmengecek pasangan A–C akan menemukan dua citra yang tidak mirip.

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

OUT = Path("/kaggle/working/leakage_out"); OUT.mkdir(parents=True, exist_ok=True)

class UF:
    def __init__(s, n): s.p = list(range(n))
    def find(s, x):
        while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def union(s, a, b):
        ra, rb = s.find(a), s.find(b)
        if ra != rb: s.p[rb] = ra

# ---------------- TAHAP 1: indeks ----------------
def stage1_index(name, cfg):
    root = Path(cfg["root"])
    files = sorted(p for p in root.rglob("*") if p.suffix.lower() in EXTS)
    rows, bad = [], 0
    for p in files:
        try:
            raw = p.read_bytes()
            with Image.open(p) as im:
                im.load(); w, h = im.size
        except Exception as e:
            bad += 1; print(f"    [corrupt] {p.name}: {e}"); continue
        r = dict(path=str(p), fname=p.name, label=label_of(p, cfg["label_mode"]),
                 width=w, height=h, bytes=len(raw), md5=hashlib.md5(raw).hexdigest())
        if cfg["label_mode"] == "nitik_filename":
            q = parse_nitik(p.stem)
            r.update(q); r["instance_id"] = f"{q['classnum']}|{q['instnum']}"
        rows.append(r)
    df = pd.DataFrame(rows).reset_index(drop=True)
    print(f"[1] {name}: {len(df)} terindeks, {bad} corrupt, {df.label.nunique()} kelas")
    df.to_csv(OUT/f"{name}_index.csv", index=False)
    return df

# ---------------- TAHAP 2: klaster via KORELASI PIKSEL ----------------
def _thumbs(df, size=64):
    X = np.zeros((len(df), size*size), dtype=np.float32)
    for k, p in enumerate(df["path"]):
        with Image.open(p) as im:
            a = np.asarray(im.convert("L").resize((size,size), Image.Resampling.LANCZOS),
                           dtype=np.float32)
        a = a - a.mean(); s = a.std()
        X[k] = (a/s if s > 1e-6 else a).ravel()
    return X

def _variants(X, size=64):
    """8 varian dihedral: menangkap duplikat yang diputar/dicerminkan."""
    A = X.reshape(-1, size, size); out = []
    for f in (False, True):
        B = A[:, :, ::-1] if f else A
        for r in range(4):
            out.append(np.rot90(B, r, axes=(1,2)).reshape(len(A), -1).copy())
    return out

def stage2_cluster(df, name, pix_corr=0.90, dihedral=True):
    """MENGAPA BUKAN pHash (diuji offline -- jangan diubah tanpa uji ulang):
    Batik = tekstur PERIODIK. Ambang median DCT pHash tidak stabil pada citra periodik.
      FALSE POSITIVE : 6/9 pasangan lolos pHash<=5 padahal korelasi ~0,0
      FALSE NEGATIVE : pasangan korelasi 1,000 berjarak pHash 9/10/18 -> TERLEWAT
    Korelasi piksel memisahkan bersih dan untuk n<=5000 dihitung EKSAK via satu matmul."""
    n = len(df)
    print(f"\n[2] {name}: near-duplicate via KORELASI PIKSEL (>= {pix_corr})"
          f"{' + dihedral' if dihedral else ''}")
    X = _thumbs(df); Vs = _variants(X) if dihedral else [X]
    d = X.shape[1]; best = np.zeros((n,n), dtype=np.float32)
    for V in Vs:
        for a in range(0, n, 1024):
            np.maximum(best[a:a+1024], (X[a:a+1024] @ V.T)/d, out=best[a:a+1024])
    best = np.maximum(best, best.T); np.fill_diagonal(best, -1.0)

    uf = UF(n)
    ii, jj = np.where(np.triu(best >= pix_corr, k=1))
    for i, j in zip(ii, jj): uf.union(int(i), int(j))
    ed = pd.DataFrame([(int(i), int(j), round(float(best[i,j]),4)) for i,j in zip(ii,jj)],
                      columns=["i","j","pix_corr"])
    df = df.copy(); df["dup_group"] = [uf.find(i) for i in range(n)]

    dup_direct = set(ed["i"]) | set(ed["j"])
    sizes = df["dup_group"].value_counts()
    n_exact = int(df["md5"].duplicated().sum())
    n_trans = int((df["dup_group"].map(sizes) > 1).sum())

    chained = 0
    for g, sub in df[df["dup_group"].map(sizes) > 1].groupby("dup_group"):
        idx = sub.index.to_numpy()
        if len(idx) < 3: continue
        m = best[np.ix_(idx, idx)]
        if float(m[~np.eye(len(idx), dtype=bool)].min()) < pix_corr: chained += 1

    print(f"    citra                       : {n}")
    print(f"    md5 identik (exact dup)     : {n_exact}")
    print(f"    citra dgn kembaran LANGSUNG : {len(dup_direct)} ({100*len(dup_direct)/n:.1f}%)  <- ANGKA UNTUK PAPER")
    print(f"    citra dlm klaster transitif : {n_trans} ({100*n_trans/n:.1f}%)  <- untuk split saja")
    print(f"    unique instances (grup)     : {len(sizes)}")
    print(f"    klaster terbesar            : {int(sizes.max())} citra")
    if chained:
        print(f"    [!] {chained} klaster BERANTAI -- dikecualikan dari angka laporan,")
        print(f"        tetap dipakai utk split (konservatif).")

    lab = df["label"].to_numpy()
    xe = ed[lab[ed["i"]] != lab[ed["j"]]].copy()
    if len(xe):
        for c, s in (("label",lab), ("fname",df["fname"].to_numpy()), ("path",df["path"].to_numpy())):
            xe[f"{c}_i"], xe[f"{c}_j"] = s[xe["i"]], s[xe["j"]]
    n_xi = len(set(xe["i"]) | set(xe["j"])) if len(xe) else 0
    print(f"    PASANGAN lintas-kelas       : {len(xe)} ({n_xi} citra terlibat)")
    if len(xe):
        print(f"    [!!] citra ~sama berlabel BEDA -> KONTAMINASI, bukan sekadar kebocoran.")
        print(f"         Sambungkan ke Sec. 3.1. WAJIB verifikasi mata sebelum diklaim.")
        xe[["fname_i","label_i","fname_j","label_j","pix_corr","path_i","path_j"]]\
          .sort_values("pix_corr", ascending=False).to_csv(OUT/f"{name}_crossclass_pairs.csv", index=False)

    ed.to_csv(OUT/f"{name}_dup_edges.csv", index=False)
    df.to_csv(OUT/f"{name}_index_grouped.csv", index=False)
    np.save(OUT/f"{name}_best.npy", best.astype(np.float16))
    return df, dict(dataset=name, n=n, exact_dup=n_exact, dup_imgs_direct=len(dup_direct),
                    dup_rate=round(100*len(dup_direct)/n,1), dup_imgs_transitive=n_trans,
                    n_groups=len(sizes), largest=int(sizes.max()), chained_clusters=chained,
                    xclass_pairs=len(xe), xclass_images=n_xi)

# ---------------- TAHAP 3: kebocoran di bawah split paper ----------------
def stratified_split(df, seed, test_size=0.30):
    """Replika split paper: 70/30 stratified per kelas, TANPA grouping."""
    rng = np.random.default_rng(seed); te = []
    for _, sub in df.groupby("label"):
        idx = sub.index.to_numpy().copy(); rng.shuffle(idx)
        te.extend(idx[:max(1, int(round(len(idx)*test_size)))].tolist())
    te = set(te)
    return np.array([i not in te for i in df.index]), np.array([i in te for i in df.index])

def stage3_leak(df, name, seeds=(0,1,2,3,4)):
    print(f"\n[3] {name}: kebocoran near-dup di bawah split 70/30 stratified paper")
    sizes = df["dup_group"].value_counts(); rows = []
    for s in seeds:
        tr_m, te_m = stratified_split(df, s)
        tr_groups = set(df.loc[tr_m, "dup_group"])
        te_idx = df.index[te_m]
        te_multi = df.loc[te_idx][df.loc[te_idx, "dup_group"].map(sizes) > 1]
        leaked = int(te_multi["dup_group"].isin(tr_groups).sum())
        rows.append(dict(seed=s, n_test=int(te_m.sum()), leaked_test=leaked,
                         leak_pct=round(100*leaked/max(1,int(te_m.sum())),1)))
        print(f"    seed {s}: {leaked}/{int(te_m.sum())} citra uji punya kembaran di train "
              f"({rows[-1]['leak_pct']}%)")
    r = pd.DataFrame(rows); r.to_csv(OUT/f"{name}_leak_under_paper_split.csv", index=False)
    print(f"    RERATA kebocoran: {r['leak_pct'].mean():.1f}% +/- {r['leak_pct'].std():.1f}")
    print(f"    -> Angka ini menggantikan caveat kualitatif Sec. 4.4.")
    return r

# ---------------- TAHAP 4: split bebas-bocor ----------------
def stage4_emit(df, name, seeds=(0,1,2,3,4), test_size=0.30):
    print(f"\n[4] {name}: menulis split GROUPED bebas-bocor")
    out = []
    for s in seeds:
        rng = np.random.default_rng(1000+s)
        gl = df.groupby("dup_group").agg(label=("label", lambda x: x.mode()[0]), n=("label","size"))
        te_groups = set()
        for lab_, sub in gl.groupby("label"):
            g = sub.index.to_numpy().copy(); rng.shuffle(g)
            target = sub["n"].sum()*test_size; acc = 0
            for gid in g:
                if acc >= target: break
                te_groups.add(gid); acc += int(gl.loc[gid,"n"])
        split = np.where(df["dup_group"].isin(te_groups), "test", "train")
        t = df[["path","fname","label","dup_group"]].copy(); t["seed"], t["split"] = s, split
        out.append(t)
        ov = len(set(df.loc[split=="train","dup_group"]) & set(df.loc[split=="test","dup_group"]))
        print(f"    seed {s}: train {int((split=='train').sum())} / test {int((split=='test').sum())}"
              f" | overlap grup = {ov} {'OK' if ov==0 else '<< BUG'}")
    pd.concat(out).to_csv(OUT/f"{name}_grouped_splits.csv", index=False)
    print(f"    -> {name}_grouped_splits.csv")

print("modul siap.")

## Sel 3 — Validasi alat pada data ground-truth sintetisSebelum mempercayai angka pada data asli, buktikan dulu alatnya benar: tanam duplikat yang**sudah diketahui**, lalu cek apakah ditemukan **persis**.Target: **recall 100%, precision 100%**. Kalau tidak, jangan lanjut.

In [ ]:
import shutil, json
FAKE = Path("/kaggle/working/fake_ds"); shutil.rmtree(FAKE, ignore_errors=True)

def _motif(seed, size=256):
    r = np.random.default_rng(seed*7919)
    base = r.normal(0,1,(8,8))
    tile = np.kron(base, np.ones((size//8, size//8)))          # periodik, mirip batik
    x, y = np.meshgrid(np.linspace(0, r.uniform(4,20)*np.pi, size),
                       np.linspace(0, r.uniform(4,20)*np.pi, size))
    g = np.sin(x + r.uniform(0,6))*np.cos(y + r.uniform(0,6))
    a = 0.6*tile + 0.4*g; a = (a-a.min())/(np.ptp(a)+1e-9)
    img = np.stack([a*r.uniform(120,255)]*3, -1)
    return Image.fromarray(np.clip(img + r.normal(0,4,img.shape),0,255).astype(np.uint8))

planted = {"exact":[], "near":[], "xclass":[]}
classes = [f"kelas_{i}" for i in range(4)]
for c in classes: (FAKE/c).mkdir(parents=True, exist_ok=True)
uid = 0
for ci, c in enumerate(classes):
    for k in range(20):
        im = _motif(ci*100+k); f0 = f"img_{uid:03d}.jpg"
        im.save(FAKE/c/f0, quality=92); uid += 1
        if k == 0:                                    # duplikat byte-identik
            f1 = f"img_{uid:03d}.jpg"; shutil.copy(FAKE/c/f0, FAKE/c/f1); uid += 1
            planted["exact"].append(tuple(sorted((f0,f1))))
        if k == 1:                                    # near-dup: resize + JPEG q55
            f1 = f"img_{uid:03d}.jpg"
            im.resize((203,203), Image.Resampling.LANCZOS).save(FAKE/c/f1, quality=55); uid += 1
            planted["near"].append(tuple(sorted((f0,f1))))
        if k == 2 and ci < 3:                         # duplikat LINTAS-KELAS
            f1 = f"img_{uid:03d}.jpg"
            im.resize((240,240), Image.Resampling.LANCZOS).save(FAKE/classes[ci+1]/f1, quality=80); uid += 1
            planted["xclass"].append(tuple(sorted((f0,f1))))

DATASETS["fake"] = dict(root=str(FAKE), label_mode="parent", expect_n=uid, expect_k=4)
_df = stage1_index("fake", DATASETS["fake"])
_df, _s = stage2_cluster(_df, "fake")

ed = pd.read_csv(OUT/"fake_dup_edges.csv"); ix = pd.read_csv(OUT/"fake_index.csv")
found = {tuple(sorted((ix.fname[r.i], ix.fname[r.j]))) for _, r in ed.iterrows()}
allp  = {tuple(sorted(t)) for v in planted.values() for t in v}
tp, fn, fp = found & allp, allp - found, found - allp
print("\n" + "="*60)
print(f"VALIDASI: ditanam {len(allp)} pasangan | terdeteksi {len(found)}")
print(f"  TP={len(tp)}  FN={len(fn)}  FP={len(fp)}")
print(f"  recall    = {len(tp)/len(allp):.0%}")
print(f"  precision = {len(tp)/max(1,len(found)):.0%}")
if fn: print("  TERLEWAT :", sorted(fn))
if fp: print("  FALSE POS:", sorted(fp))
print("="*60)
assert len(fn) == 0 and len(fp) == 0, "VALIDASI GAGAL -- jangan percaya angka pada data asli!"
print("LULUS. Alat boleh dipakai pada data asli.")

## Sel 4 — Jalankan pada data asliNitik dijalankan lebih dulu karena berfungsi sebagai **validasi kedua pada batik asli**:rotasi 90/180/270 tersimpan sebagai file terpisah, jadi pencocokan dihedral **harus** menemukantepat 3 kembaran untuk tiap citra dasar. Kalau grup duplikat yang ditemukan cocok dengan`instance_id` dari nama file, alat ini terbukti bekerja pada tekstur batik sungguhan — bukanhanya pada data sintetis.> **Nitik: kebocoran rotasi sudah pernah diuji** (LEAKY − GROUP = 0). Angka kebocoran Nitik di> sini akan tinggi **secara desain** (rotasi memang duplikat dihedral) dan itu **bukan temuan> baru** — jangan laporkan sebagai temuan. Gunanya di sini murni sebagai uji alat.

In [ ]:
SEEDS = (0,1,2,3,4)
results, summaries = {}, []

for name in ["nitik", "dion", "jambi"]:
    cfg = DATASETS[name]
    if not Path(cfg["root"]).exists():
        print(f"\n!! LEWATI {name}: path tidak ada -> {cfg['root']}\n"); continue
    print("\n" + "="*78); print(f"### {name.upper()}"); print("="*78)
    df = stage1_index(name, cfg)
    df, s = stage2_cluster(df, name, pix_corr=0.90, dihedral=True)
    summaries.append(s)
    stage3_leak(df, name, SEEDS)
    stage4_emit(df, name, SEEDS)
    results[name] = df

# --- validasi kedua: apakah grup dup Nitik == instance_id dari nama file? ---
if "nitik" in results:
    d = results["nitik"]
    if "instance_id" in d.columns:
        ct = d.groupby("dup_group")["instance_id"].nunique()
        purity = (ct == 1).mean()
        cover = d.groupby("instance_id")["dup_group"].nunique()
        print("\n" + "="*78)
        print("VALIDASI KEDUA (batik asli) — grup duplikat vs instance_id dari nama file")
        print(f"  grup yang isinya satu instance saja : {purity:.1%}  (harap ~100%)")
        print(f"  instance yang utuh dalam satu grup  : {(cover==1).mean():.1%}  (harap ~100%)")
        if purity > 0.95 and (cover==1).mean() > 0.95:
            print("  -> Deteksi dihedral menemukan kembali struktur rotasi Nitik.")
            print("     Alat terbukti bekerja pada tekstur batik sungguhan.")
        else:
            print("  -> TIDAK cocok. Periksa parser nama file / ambang korelasi sebelum")
            print("     mempercayai angka DION & Jambi.")
        print("="*78)

## Sel 5 — Ringkasan & berkas keluaranTabel di bawah adalah bahan mentah untuk mengganti caveat kualitatif §4.4.

In [ ]:
summ = pd.DataFrame(summaries)
summ.to_csv(OUT/"ALL_summary.csv", index=False)
print("RINGKASAN AUDIT KEBOCORAN"); print("="*78)
print(summ.to_string(index=False))

leaks = []
for n in summ["dataset"]:
    f = OUT/f"{n}_leak_under_paper_split.csv"
    if f.exists():
        r = pd.read_csv(f); leaks.append(dict(dataset=n,
            leak_mean=round(r.leak_pct.mean(),1), leak_std=round(r.leak_pct.std(),1)))
if leaks:
    lk = pd.DataFrame(leaks); lk.to_csv(OUT/"ALL_leak.csv", index=False)
    print("\nKEBOCORAN DI BAWAH SPLIT 70/30 PAPER (% citra uji berkembaran di train)")
    print("="*78); print(lk.to_string(index=False))

print("\nBERKAS DI /kaggle/working/leakage_out :")
for f in sorted(OUT.iterdir()):
    print(f"   {f.name:45s} {f.stat().st_size/1024:8.1f} KB")

print("""
LANGKAH BERIKUTNYA
------------------
1. Unduh seluruh isi leakage_out/ (Output -> Download).
2. BUKA *_crossclass_pairs.csv dan VERIFIKASI MATA tiap pasangan (kolom path_i/path_j).
   Korelasi 0,90 itu heuristik, bukan kebenaran. Jangan klaim tanpa melihat citranya.
3. Untuk paper, pakai kolom dup_imgs_direct / dup_rate -- BUKAN dup_imgs_transitive.
4. Bila laju kebocoran DION/Jambi tinggi: pertimbangkan menjalankan ulang Probe B memakai
   *_grouped_splits.csv, lalu bandingkan akurasi COLOR paper-split vs grouped-split.
   Kalau selisihnya kecil, klaim Sec. 4.4 TERBUKTI -- dengan data, bukan asumsi.
5. Kirim CSV-nya untuk dibahas: apakah jadi sub-bagian sendiri atau memperkuat Sec. 4.4.
""")